## Które cząsteczki chcemy tworzyć

Ze względu na ograniczenia VeloxChema chcemy ograniczyć się tylko do tych związków kleszczowych, które zawierająca pierwsiatki z bloku d z 1 okresu 

In [1]:
import pandas as pd

df = pd.read_csv('tmqm_Xall.csv') #zczytana pełna baza tmQM w postaci pliku .csv
df.describe()

,num_atoms,q,S,MND,Electronic_E,Dispersion_E,Dipole_M,Metal_q,HL_Gap,HOMO_Energy,LUMO_Energy,Polarizability
count,108541.000000,108541.000000,108541.0,108541.000000,108541.000000,108541.000000,108541.000000,108541.000000,108541.000000,108541.000000,108541.000000,108541.000000
mean,66.495287,0.132374,0.0,5.632609,-2987.303526,-0.142228,5.799450,0.162882,0.108504,-0.197593,-0.089089,397.124993
std,27.145682,0.399397,0.0,2.077392,1634.083288,0.070115,3.927489,0.782914,0.033575,0.053825,0.054517,152.460083
min,7.000000,-2.000000,0.0,1.000000,-29008.530471,-1.122263,0.000000,-3.083370,0.001570,-0.442030,-0.371990,51.249960
25%,47.000000,0.000000,0.0,4.000000,-3551.761675,-0.180599,2.977000,-0.235330,0.087780,-0.210770,-0.107840,285.598661
50%,63.000000,0.000000,0.0,6.000000,-2682.286991,-0.129635,5.390300,0.249460,0.109320,-0.186870,-0.079030,372.962979
75%,82.000000,0.000000,0.0,6.000000,-1957.796717,-0.090185,8.211900,0.661090,0.129570,-0.168440,-0.056580,483.132746
max,569.000000,1.000000,0.0,19.000000,-295.243567,-0.005205,106.464600,2.330130,0.307420,0.066540,0.198190,3002.513834


Na samym początku zidentyfikujmy izomery

In [ ]:
df.atom_types.value_counts().values #origin_ID jest unikalne, po składzie atomowym widzimy, że baza zawiera izomery

array([20, 13, 12, ...,  1,  1,  1], shape=(106761,))

In [3]:
conf = df['atom_types'].value_counts()
indexes = conf[conf > 1].index.tolist()

len(indexes)

1465

In [ ]:
#do czego sie sprowadzaja te izomery?

c1 = df[df['atom_types'] == indexes[0]]

c1.iloc[0] == c1.iloc[1] #ten sam skład atomowy, ale inne współrzędne i co za tym idzie, inne niektóre właściwości

#z naszej perspektywy nie ma potrzeby usuwać tego typu danych

num_atoms          True
atom_types         True
atom_coords       False
origin_ID         False
q                  True
S                  True
Stoichiometry      True
MND                True
ID                False
CSD_code          False
Electronic_E      False
Dispersion_E      False
Dipole_M          False
Metal_q           False
HL_Gap            False
HOMO_Energy       False
LUMO_Energy       False
Polarizability    False
CSD_years          True
SMILES             True
dtype: bool

In [6]:
#warto jeszcze zwrócić uwagę, na ładunek jonowy - widzimy to w kolumnie Stoichiometry

print(len(df[df.Stoichiometry.str.contains('+', regex=False)]))
print(len(df[df.Stoichiometry.str.contains('-', regex=False)]))

16791
2422


### Filtracja tmQM ze względu na skład atomowy

In [8]:
from ase.data import atomic_numbers

def is_elements_legal_check(atom_types):

    legal_elements = [1,5,6,7,8,9,13,14,15,16,17,21,22,23,24,25,26,27,28,29,30,33,34,35] #to są liczby atomowe

    l = [atomic_numbers[at] for at in atom_types]

    for el in l:
        if el not in legal_elements:
            #print(f'Element {el} is not legal!')
            return el
    return 0


l = []
bad_csd_code =[]
for row in range(len(df)):
    sanity_check = is_elements_legal_check(eval(df.atom_types.iloc[row]))
    if sanity_check != 0:
        l.append(sanity_check)
        bad_csd_code.append(df.CSD_code.iloc[row])


print(len(l))
pd.Series(l).value_counts().head() #l zawiera pierwszy pierwiastek w czasteczce, który wyklucza ją z dalszych rozwazań



68429


46    10422
44     9688
78     8216
77     5206
45     4528
Name: count, dtype: int64

In [9]:
len(list(set(bad_csd_code))) #tyle jest kompleksów, których nie możemy policzyć

68429

In [10]:
#czyli zostaje nam

good_metals = df[~df.CSD_code.isin(bad_csd_code)]
good_metals.describe()

,num_atoms,q,S,MND,Electronic_E,Dispersion_E,Dipole_M,Metal_q,HL_Gap,HOMO_Energy,LUMO_Energy,Polarizability
count,40112.000000,40112.000000,40112.0,40112.000000,40112.000000,40112.000000,40112.000000,40112.000000,40112.000000,40112.000000,40112.000000,40112.000000
mean,66.158382,0.110914,0.0,5.468389,-3756.358330,-0.136256,5.442145,0.311210,0.100333,-0.189209,-0.088875,383.727690
std,27.276953,0.393838,0.0,1.782563,1651.020888,0.067363,4.038683,0.773508,0.036918,0.054554,0.055227,150.305891
min,9.000000,-2.000000,0.0,2.000000,-27194.949918,-1.122263,0.000000,-3.083370,0.001570,-0.438820,-0.354380,51.249960
25%,47.000000,0.000000,0.0,4.000000,-4148.906555,-0.172654,2.595975,0.088938,0.075150,-0.208025,-0.106980,275.001111
50%,62.000000,0.000000,0.0,5.000000,-3395.783409,-0.123486,4.973350,0.496910,0.100665,-0.181040,-0.081270,357.771936
75%,81.000000,0.000000,0.0,6.000000,-2821.302688,-0.086600,7.726675,0.800097,0.123980,-0.159500,-0.058020,468.828138
max,569.000000,1.000000,0.0,14.000000,-1242.449904,-0.005205,106.464600,1.679860,0.280170,0.066540,0.155110,3002.513834


In [ ]:
print(len(good_metals[good_metals.Stoichiometry.str.contains('+', regex=False)]))
print(len(good_metals[good_metals.Stoichiometry.str.contains('-', regex=False)]))

print(len(good_metals[good_metals.Stoichiometry.str.contains('1+', regex=False)]))
print(len(good_metals[good_metals.Stoichiometry.str.contains('2-', regex=False)])) #do wywalenia

5581
1131
5581
1


In [12]:
good_metals[good_metals.Stoichiometry.str.contains('2-')]

,num_atoms,atom_types,atom_coords,origin_ID,q,S,Stoichiometry,MND,ID,CSD_code,Electronic_E,Dispersion_E,Dipole_M,Metal_q,HL_Gap,HOMO_Energy,LUMO_Energy,Polarizability,CSD_years,SMILES
3305,47,"['C', 'C', 'C', 'C', 'C', 'H', 'C', 'C', 'C', ...","[['8.24092964522156', '11.8117129330756', '7.5...",IKEJAJ,-2,0,C20H16CrN6S4(2-),6,tmqm_003305,IKEJAJ,-3736.589013,-0.096317,16.4399,-0.03505,0.01621,0.06654,0.08275,373.129308,2020-2024,NaN


In [13]:
good_metals = good_metals[~good_metals.Stoichiometry.str.contains('2-')]
good_metals.describe()

,num_atoms,q,S,MND,Electronic_E,Dispersion_E,Dipole_M,Metal_q,HL_Gap,HOMO_Energy,LUMO_Energy,Polarizability
count,40111.000000,40111.000000,40111.0,40111.000000,40111.000000,40111.000000,40111.000000,40111.000000,40111.000000,40111.000000,40111.000000,40111.000000
mean,66.158859,0.110967,0.0,5.468375,-3756.358823,-0.136257,5.441871,0.311219,0.100336,-0.189215,-0.088880,383.727954
std,27.277126,0.393702,0.0,1.782583,1651.041466,0.067364,4.038360,0.773516,0.036917,0.054540,0.055221,150.307756
min,9.000000,-1.000000,0.0,2.000000,-27194.949918,-1.122263,0.000000,-3.083370,0.001570,-0.438820,-0.354380,51.249960
25%,47.000000,0.000000,0.0,4.000000,-4148.916895,-0.172654,2.595950,0.088985,0.075150,-0.208030,-0.106980,274.998255
50%,62.000000,0.000000,0.0,5.000000,-3395.783347,-0.123488,4.973300,0.496920,0.100670,-0.181040,-0.081270,357.771545
75%,81.000000,0.000000,0.0,6.000000,-2821.297775,-0.086599,7.726450,0.800125,0.123980,-0.159500,-0.058020,468.828277
max,569.000000,1.000000,0.0,14.000000,-1242.449904,-0.005205,106.464600,1.679860,0.280170,0.048530,0.155110,3002.513834


In [35]:
good_metals.q.unique()

array([ 1,  0, -1])

In [14]:
import ast

good_metals['atom_types'] = good_metals['atom_types'].apply(ast.literal_eval)

In [19]:
#tabela czestosci wystepowania konkretnych pierwiastków w finalnych dfie

from collections import Counter

atom_counter = Counter()

for i,lista in enumerate(good_metals['atom_types']):

    atom_counter.update(lista)

for atom, c in atom_counter.items():
    print(atom, c)

Sc 469
C 1078805
H 1229486
B 7596
N 126436
O 94899
P 17055
Si 5128
S 17213
F 16002
Cl 14629
Se 766
Br 5350
Ti 3235
As 222
V 1429
Cr 2080
Mn 2245
Fe 5534
Co 3781
Ni 10695
Cu 3487
Zn 7156


### Ile w finalnym zbiorze jest kompleksów z originalnego tmQM_X1.xyz? (dzielenie się zadaniami)

In [ ]:
with open('tmQM_X1.xyz', 'r') as f:
    lines = f.readlines()


ids_from_X1 = [x.split('|')[0].replace('CSD_code = ', '').strip() for x in lines if 'CSD_code' in x]
print(ids_from_X1[:3])

len(good_metals[good_metals.origin_ID.isin(ids_from_X1)])  #około 1/3 więc możemy zostać przy podziale na pliki X1, (X2, X3)

['WELROW', 'VUCVUN', 'QIWWAY']


13158